In [1]:
import os, sys, mne, socket, time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from nilearn import plotting

from initialization import load_paths

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
subjID = 99
day = 3
p = load_paths(subjID, day)
montage = mne.channels.make_standard_montage("easycap-M1")

In [3]:
# List all the files in the directory
f_list = os.listdir(p['EEGfiles'])
f_list = [f for f in f_list if f.endswith('.vhdr')]
if len(f_list) == 1:
    raw_data = mne.io.read_raw_brainvision(p['EEGfiles'] + '/' + f_list[0], preload=True)
    raw_data.drop_channels(['LM', 'RM'])
    raw_data.set_montage(montage)

Extracting parameters from /d/DATD/datd/TMS_ori/EEGfiles/sub99/day03/sub99_day03.vhdr...
Setting channel info structure...
Reading 0 ... 4382679  =      0.000 ...  4382.679 secs...


In [ ]:
raw_data.compute_psd(fmax=55, average='mean', n_fft=2048).plot()
plt.show()

In [4]:
raw_data = mne.add_reference_channels(raw_data, ref_channels='Cz')

/var/folders/cf/d9j3_ppd5wb4bn9y_fgj3nb800010l/T/ipykernel_92442/1662345521.py:1: RuntimeWarning: The locations of multiple reference channels are ignored.
  raw_data = mne.add_reference_channels(raw_data, ref_channels='Cz')


In [5]:
events, tah = mne.events_from_annotations(raw_data)
events_dict = {
    'BlockOnset': 1001,
    'BlockEnd': 7,
    'Fixation': 1,
    'Delay1': 2,
    'Delay2': 3,
    'Response': 4,
    'Feedback': 5,
    'ITI': 6,
}
trl_events = {
    'inPF': 11,
    'outPF': 10,
}

Used Annotations descriptions: ['Response/R  1', 'Stimulus/S  1', 'Stimulus/S  2', 'Stimulus/S  3', 'Stimulus/S  4', 'Stimulus/S  5', 'Stimulus/S  6', 'Stimulus/S  7', 'Stimulus/S 10', 'Stimulus/S 11']


In [ ]:
block_onset_id = events_dict['BlockOnset']
block_end_id = events_dict['BlockEnd']
# Find the start and end events
start_events = events[events[:, 2] == block_onset_id]
end_events = events[events[:, 2] == block_end_id]

# Create custom epochs
block_epochs_list = []
for start_event in start_events:
    # Find the next end event after the current start event
    end_event = end_events[end_events[:, 0] > start_event[0]][0]

    # Define the time window for the block epoch
    tmin = (start_event[0] - raw_data.first_samp) / raw_data.info['sfreq']
    tmax = (end_event[0] - raw_data.first_samp) / raw_data.info['sfreq']

    # Create the block epoch
    block = raw_data.copy().crop(tmin, tmax)
    block_epochs_list.append(block)

In [ ]:
# Apply a highpass filter to each epoch
block_epochs_list = [block.filter(l_freq=0.1, h_freq=None) for block in block_epochs_list]

In [ ]:
block_epochs_list[1].times

In [10]:
startBlock_event

array([27901,     0,  1001])

In [11]:
endBlock_event

array([353617,      0,      7])

In [28]:
block_onset_id = events_dict['BlockOnset']
block_end_id = events_dict['BlockEnd']
# Find the start and end events
startBlock_events = events[events[:, 2] == block_onset_id]
endBlock_events = events[events[:, 2] == block_end_id]
sampBuffer = 1000
# Create custom epochs
# block_epochs_list = []
trials_list = []
for blkIdx, startBlock_event in enumerate(startBlock_events):
    print(f"Running block {blkIdx + 1} of {len(startBlock_events)}")
    # Find the next end event after the current start event
    endBlock_event = endBlock_events[endBlock_events[:, 0] > startBlock_event[0]][0]

    # Create events_block to be used for trial definition
    events_block = events[(events[:, 0] > startBlock_event[0]) & (events[:, 0] < endBlock_event[0])]

    # Define the time window for the block epoch and epoch the block
    tminBlock = (startBlock_event[0] - raw_data.first_samp) / raw_data.info['sfreq']
    tmaxBlock = (endBlock_event[0] - raw_data.first_samp) / raw_data.info['sfreq']
    block = raw_data.copy().crop(tminBlock, tmaxBlock)
    # Apply a highpass filter to the block
    block.filter(l_freq=0.1, h_freq=None, verbose=False)

    # Find the start and end events for each trial
    startTrial_events = events_block[events_block[:, 2] == events_dict['Fixation']]
    endTrial_events = events_block[events_block[:, 2] == events_dict['Response']]

    # for startTrial_event in startTrial_events:
    for trlIdx, startTrial_event in enumerate(startTrial_events):
        print(f"    Running trial {trlIdx + 1} of {len(startTrial_events)}")
        # Find the next end event after the current start event
        endTrial_event = endTrial_events[endTrial_events[:, 0] > startTrial_event[0]][0]

        # Define the time window for the trial epoch
        tminTrial = (startTrial_event[0] - sampBuffer - block.first_samp) / block.info['sfreq']
        tmaxTrial = (endTrial_event[0] + sampBuffer - block.first_samp) / block.info['sfreq']

        # Create the trial epoch
        trial = block.copy().crop(tminTrial, tmaxTrial)
        trials_list.append(trial)

Running block 1 of 13
    Running trial 1 of 24
    Running trial 2 of 24
    Running trial 3 of 24
    Running trial 4 of 24
    Running trial 5 of 24
    Running trial 6 of 24
    Running trial 7 of 24
    Running trial 8 of 24
    Running trial 9 of 24
    Running trial 10 of 24
    Running trial 11 of 24
    Running trial 12 of 24
    Running trial 13 of 24
    Running trial 14 of 24
    Running trial 15 of 24
    Running trial 16 of 24
    Running trial 17 of 24
    Running trial 18 of 24
    Running trial 19 of 24
    Running trial 20 of 24
    Running trial 21 of 24
    Running trial 22 of 24
    Running trial 23 of 24
    Running trial 24 of 24
Running block 2 of 13
    Running trial 1 of 24
    Running trial 2 of 24
    Running trial 3 of 24
    Running trial 4 of 24
    Running trial 5 of 24
    Running trial 6 of 24
    Running trial 7 of 24
    Running trial 8 of 24
    Running trial 9 of 24
    Running trial 10 of 24
    Running trial 11 of 24
    Running trial 12 of 24
  

In [23]:
# Plot ERP across all trials do it manually
plt.figure()

mean_data = np.mean([trial.get_data() for trial in trials_list], axis=0)
plt.plot(trials_list[0].times, mean_data[0, 0, :])
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (uV)')
plt.title('ERP across all trials')
plt.show()

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (312, 64) + inhomogeneous part.

<Figure size 640x480 with 0 Axes>

In [18]:
block.info['sfreq']

1000.0

In [ ]:
# Get variance of electrodes across whole recording
elec_var = raw_data.get_data().var(axis=1)

In [ ]:
# Plot data for block 0 for selected time window
block_epochs_list[0].plot(n_channels=64, scalings=dict(eeg=20e-5), title='Block 0')
plt.show()

In [ ]:
# Print duration of the epochs
total_dur = 0
for epoch in epochs_list:
    total_dur += epoch.times[-1]
    print(f'Epoch duration: {epoch.times[-1]:.2f} s')
print(f'Total duration: {total_dur:.2f} s')

In [ ]:
# Re-epoch data